# Wake convergence example

このNotebookでは、1つのOpenVSPモデルに対してVSPAEROの安定微係数解析を繰り返し実行し、wakeの**反復収束**と**空間離散化収束**を順番に確認する。

2変数の総当たりは行わない。

1. `NumWakeNodes`を固定して`WakeNumIter`を変える。
2. 十分な`WakeNumIter`を固定して`NumWakeNodes`を変える。
3. 2つのsweepが共有するcaseは1回だけ実行する。
4. 全caseを`wake_convergence.csv`へ保存する。

`WakeNumIter`と`NumWakeNodes`はVSPAEROで別のsolver settingであり、それぞれ反復回数とwakeの流れ方向離散化を表す。
この実行Notebookは`vsp.STABILITY_ADJOINT`を明示してwake収束を評価する。有限差分へ戻す場合は設定セルの`stability_type`だけを`vsp.STABILITY_DEFAULT`へ変更する。


## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/AnalysisVSPAERO.py`を含むリポジトリルートを見つける。

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import shutil
import sys
import time

import pandas as pd
import openvsp as vsp
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "AnalysisVSPAERO.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/AnalysisVSPAERO.py. "
        "Run this notebook inside the repository or set repo_root manually."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.AnalysisVSPAERO import (  # noqa: E402
    validate_vsp3_for_stability_derivatives,
    vsp_stability_derivatives,
)
from src.TrimTurnSolver import (  # noqa: E402
    read_vspaero_stab,
    resolve_control_columns_from_stab,
)
from src.util import workdir  # noqa: E402

print("repo_root:", repo_root)

## 2. 入力モデルと収束試験条件

通常変更する設定はこのセルに集約する。

`wake_iters`は反復収束試験、`wake_nodes`は空間離散化収束試験で使う。交点のcaseは両方のsweepに使うが、実行は1回だけにする。

In [ ]:
# model_name = "Boeing_777-9x_mod"
model_name = "G103A"

vsp3_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)
output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "wake_convergence"
)
results_path = output_dir / "wake_convergence.wbhv.csv"

# Phase A: iteration convergence at a moderately fine wake discretization.
wake_iters = [3, 5, 8, 12, 16]
wake_nodes_for_iter_sweep = 32

# Phase B: wake-node convergence at a sufficiently large iteration count.
wake_nodes = [8, 16, 32, 64]
wake_iter_for_node_sweep = 12

alpha_deg = 2.0
mach = 0.1
reynolds = 4.4e6
ncpu = 8
# stability_type = vsp.STABILITY_ADJOINT
stability_type = vsp.STABILITY_DEFAULT

fixed_wake_flag = None
redirect_file = ""
verbose = 1
vspaero_verbose = 0

output_dir.mkdir(parents=True, exist_ok=True)

if not vsp3_path.exists():
    raise FileNotFoundError(vsp3_path)

print("vsp3_path:", vsp3_path)
print("output_dir:", output_dir)
print("stability_type:", stability_type)
print("wake_iters:", wake_iters, "at NumWakeNodes =", wake_nodes_for_iter_sweep)
print("wake_nodes:", wake_nodes, "at WakeNumIter =", wake_iter_for_node_sweep)

## 3. 元モデルの事前検証

全caseで同じモデルを使うため、VSPAEROの保存済みgeometry selection、参照値、CG、Symmetryを元モデルに対して1回だけ検証する。Set名、Geom名、Control Surface Group名は解析の必須条件にしない。


In [ ]:
validation = validate_vsp3_for_stability_derivatives(
    vsp3_path,
    verbose=2,
)

print("passed:", validation["passed"])
print("errors:", validation["errors"])
print("warnings:", validation["warnings"])

if not validation["passed"]:
    raise RuntimeError(validation["errors"])

## 4. case準備と`.stab`読込

主ループを読みやすくするため、caseディレクトリ準備、`.stab`探索、`.stab`の1行化だけを補助関数として残す。

In [ ]:
AERO_COEFFICIENTS = [
    "CL",
    "CD",
    "CS",
    "CMl",
    "CMm",
    "CMn",
]


def prepare_case_vsp3(case_dir: Path, source_vsp3_path: Path) -> Path:
    case_dir.mkdir(parents=True, exist_ok=True)
    case_vsp3_path = case_dir / source_vsp3_path.name

    for path in case_dir.glob(f"{case_vsp3_path.stem}*"):
        if path.is_file() and path.suffix.lower() != ".vsp3":
            path.unlink()

    shutil.copy2(source_vsp3_path, case_vsp3_path)
    return case_vsp3_path


def find_stab_file(
    case_dir: Path,
    preferred_stem: str,
    stab_file_suffix: str,
) -> Path | None:
    preferred_path = case_dir / f"{preferred_stem}{stab_file_suffix}"
    if preferred_path.exists():
        return preferred_path

    candidates = sorted(
        case_dir.glob(f"*{stab_file_suffix}"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


def flatten_stab_result(stab) -> dict[str, object]:
    control_columns = resolve_control_columns_from_stab(stab)

    row: dict[str, object] = {
        "Sref": float(stab.Sref),
        "Cref": float(stab.Cref),
        "Bref": float(stab.Bref),
        "Vinf": float(stab.V0),
        "rho": math.nan if stab.rho0 is None else float(stab.rho0),
        "AoA_deg": stab.base_condition.get("AoA"),
        "Beta_deg": stab.base_condition.get("Beta"),
        "control_groups": repr(stab.control_groups),
    }

    for delta_name in ("delta_a", "delta_e", "delta_r"):
        row[f"control_column_{delta_name}"] = control_columns.get(delta_name, "")

    for coefficient_name in AERO_COEFFICIENTS:
        if coefficient_name not in stab.derivatives.index:
            continue

        derivative_row = stab.derivatives.loc[coefficient_name]

        for derivative_name, value in derivative_row.items():
            row[f"{coefficient_name}_{derivative_name}"] = float(value)

        for delta_name, derivative_column in control_columns.items():
            if derivative_column in derivative_row.index:
                row[f"{coefficient_name}_{delta_name}"] = float(
                    derivative_row[derivative_column]
                )

    return row

## 5. 実行caseの作成

iteration sweepとwake-node sweepの和集合だけを実行する。`WakeNumIter=12, NumWakeNodes=32`の交点は両方のsweepに属するが1回だけ計算する。

In [ ]:
cases: list[tuple[int, int]] = []

for wake_iter in wake_iters:
    cases.append((wake_iter, wake_nodes_for_iter_sweep))

for wake_node_count in wake_nodes:
    case = (wake_iter_for_node_sweep, wake_node_count)
    if case not in cases:
        cases.append(case)

print("cases:")
for wake_iter, wake_node_count in cases:
    print(f"  WakeNumIter={wake_iter:3d}, NumWakeNodes={wake_node_count:3d}")

## 6. Wake convergence sweep

各caseについて、入力モデルのコピー、VSPAERO実行、`.stab`読込、結果整理を上から順に行う。

In [ ]:
rows: list[dict[str, object]] = []
sweep_start = time.perf_counter()

for case_index, (wake_iter, wake_node_count) in enumerate(cases, start=1):
    case_name = f"wake_i{wake_iter:03d}_n{wake_node_count:03d}"
    case_dir = output_dir / case_name
    case_vsp3_path = prepare_case_vsp3(case_dir, vsp3_path)

    print(
        f"[{case_index}/{len(cases)}] "
        f"WakeNumIter={wake_iter}, NumWakeNodes={wake_node_count}",
        flush=True,
    )
    case_start = time.perf_counter()

    with workdir(case_dir):
        report = vsp_stability_derivatives(
            case_vsp3_path.name,
            alpha=alpha_deg,
            mach=mach,
            reynolds=reynolds,
            stability_type=stability_type,
            ncpu=ncpu,
            wake_num_iter=wake_iter,
            wake_num_nodes=wake_node_count,
            fixed_wake_flag=fixed_wake_flag,
            redirect_file=redirect_file,
            stop_before_run=False,
            vspaero_verbose=vspaero_verbose,
            verbose=verbose,
        )

    effective_settings = report.get("vspaero_settings", {})
    effective_wake_iter = effective_settings.get("effective_wake_num_iter")
    effective_wake_nodes = effective_settings.get("effective_wake_num_nodes")
    if report.get("passed", False) and (
        effective_wake_iter != wake_iter
        or effective_wake_nodes != wake_node_count
    ):
        raise RuntimeError(
            "VSPAERO did not use the requested wake settings: "
            f"requested=({wake_iter}, {wake_node_count}), "
            f"effective=({effective_wake_iter}, {effective_wake_nodes})"
        )

    timing = report.get("timing", {})
    stab_file_suffix = effective_settings.get(
        "effective_stab_file_suffix",
        effective_settings.get("stab_file_suffix", ".stab"),
    )
    stab_path = find_stab_file(
        case_dir,
        case_vsp3_path.stem,
        stab_file_suffix,
    )

    row: dict[str, object] = {
        "wake_iter": int(wake_iter),
        "wake_nodes": int(wake_node_count),
        "in_wake_iter_sweep": (
            wake_node_count == wake_nodes_for_iter_sweep
            and wake_iter in wake_iters
        ),
        "in_wake_node_sweep": (
            wake_iter == wake_iter_for_node_sweep
            and wake_node_count in wake_nodes
        ),
        "passed": bool(report.get("passed", False)),
        "case_dir": str(case_dir),
        "vsp3_path": str(case_vsp3_path),
        "stab_path": "" if stab_path is None else str(stab_path),
        "errors": repr(report.get("errors", [])),
        "warnings": repr(report.get("warnings", [])),
        "elapsed_s": time.perf_counter() - case_start,
        "total_elapsed_s": timing.get("total_elapsed_s", math.nan),
        "compute_geometry_elapsed_s": timing.get(
            "compute_geometry_elapsed_s",
            math.nan,
        ),
        "vspaero_sweep_elapsed_s": timing.get(
            "vspaero_sweep_elapsed_s",
            math.nan,
        ),
        "result_extraction_elapsed_s": timing.get(
            "result_extraction_elapsed_s",
            math.nan,
        ),
    }

    if row["passed"] and stab_path is not None:
        stab = read_vspaero_stab(stab_path)
        row.update(flatten_stab_result(stab))

    rows.append(row)

    print(
        f"    passed={row['passed']}, "
        f"elapsed={row['elapsed_s']:.1f} s, "
        f"stab={row['stab_path']}",
        flush=True,
    )

results = pd.DataFrame(rows).sort_values(
    ["wake_nodes", "wake_iter"]
).reset_index(drop=True)
results.to_csv(results_path, index=False)

print(f"Done. elapsed={time.perf_counter() - sweep_start:.1f} s")
print("results_path:", results_path)

## 7. 結果の要約

2つのsweepを別々に表示し、成功可否と計算時間を確認する。収束判定そのものはplot Notebookで行う。

In [ ]:
summary_columns = [
    "wake_iter",
    "wake_nodes",
    "passed",
    "elapsed_s",
    "vspaero_sweep_elapsed_s",
    "CL_Base",
    "CD_Base",
    "CS_Base",
    "CMm_Base",
]
available_summary_columns = [
    column for column in summary_columns if column in results.columns
]

print("WakeNumIter sweep")
display(
    results.loc[
        results["in_wake_iter_sweep"].astype(bool),
        available_summary_columns,
    ].sort_values("wake_iter")
)

print("NumWakeNodes sweep")
display(
    results.loc[
        results["in_wake_node_sweep"].astype(bool),
        available_summary_columns,
    ].sort_values("wake_nodes")
)

failed_cases = results.loc[
    ~results["passed"].astype(bool),
    ["wake_iter", "wake_nodes", "errors", "warnings"],
]

if failed_cases.empty:
    print("All cases passed.")
else:
    print("Failed cases:")
    display(failed_cases)